# Groq Note Summarization on Kaggle (CPU)

Runs the MIRROR Groq summarization pipeline from a Kaggle CPU notebook.
Kaggle's IP is clean and has low latency to Groq's cloud — no home-IP throttling.

**Inputs (upload to a private Kaggle dataset):**
- `notes_text_mimic3.pkl` — cleaned note texts (from `extract_notes.py`)
- `user_config.yaml` — must contain a `groq_api_keys` list

**Output (download from Output tab):**
- `note_summaries_groq_intent.json` — checkpoint with all summaries (approach A)

Drop the downloaded JSON into `data/processed/` then run locally:
```
python src/embeddings/summarize_notes_groq.py --skip_groq
```
to do the ClinicalBERT encoding step.

**Settings:** Enable Internet in the notebook sidebar before running.
Runtime: ~3-6 hours on CPU (12,000 notes at 60 RPM with key rotation).

In [ ]:
import glob, shutil, os
import requests

os.makedirs('data/processed', exist_ok=True)

# Locate uploaded files from /kaggle/input/
for fname in ['notes_text_mimic3.pkl', 'user_config.yaml']:
    hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
    if not hits:
        raise FileNotFoundError(f'{fname} not found in /kaggle/input — upload it to your dataset')
    shutil.copy(hits[0], f'data/processed/{fname}')
    print(f'Copied {hits[0]} -> data/processed/{fname}')

# Quick internet check
try:
    r = requests.get('https://api.groq.com', timeout=5)
    print(f'Groq reachable: HTTP {r.status_code}')
except Exception as e:
    print(f'WARNING: Cannot reach Groq API: {e}')
    print('Make sure Internet is enabled in the notebook sidebar!')

In [ ]:
# ── Groq summarization client (inlined from src/embeddings/summarize_notes_groq.py) ──

import json, pickle, queue as _queue, re as _re, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import numpy as np
import yaml

GROQ_API_URL = 'https://api.groq.com/openai/v1/chat/completions'
GROQ_MODEL   = 'llama-3.3-70b-versatile'
NOTE_MAX_CHARS      = 2_000
CHECKPOINT_INTERVAL = 50

_MIMIC_SECTION_PRIORITY = [
    r'brief\s+hospital\s+course', r'hospital\s+course',
    r'history\s+of\s+present\s+illness', r'pertinent\s+results?',
    r'discharge\s+diagnos(?:is|es)', r'allergies?',
    r'assessment(?:\s+and\s+plan)?', r'past\s+medical\s+history',
]

NOTE_CLINICAL_INTENT_PROMPT = """You are a clinical NLP assistant. Summarize the key clinical problems from this hospital discharge note.

INCLUDE:
- Active diagnoses and medical conditions with severity (e.g., "HFrEF EF 20%, NYHA III")
- Relevant lab values and physiological findings driving clinical decisions
- Drug allergies and medication intolerances (e.g., "penicillin allergy — anaphylaxis")
- Treatment requirements noted in the hospital course (e.g., "required IV pressors", "insulin-dependent", "failed oral antibiotics")
- Organ dysfunction affecting treatment (renal, hepatic, cardiac function)

EXCLUDE:
- Specific medication doses, frequencies, or structured prescription lists
- Social history, family history, insurance, administrative content
- Discharge instructions

Format: 4 to 8 bullet points, each under 25 words.
If no clinical data can be identified, respond: "No clinical data identified."

DISCHARGE NOTE:
{note_text}
"""


def _extract_key_sections(note_text, max_chars=NOTE_MAX_CHARS):
    if len(note_text) <= max_chars:
        return note_text
    header_pat = _re.compile(
        r'(?:^|\n)\s*(' + '|'.join(_MIMIC_SECTION_PRIORITY) + r')\s*:',
        _re.IGNORECASE,
    )
    matches = list(header_pat.finditer(note_text))
    if not matches:
        return note_text[:max_chars]

    def _pri(header):
        name = header.strip().lower()
        for i, pat in enumerate(_MIMIC_SECTION_PRIORITY):
            if _re.search(pat, name, _re.IGNORECASE):
                return i
        return len(_MIMIC_SECTION_PRIORITY)

    sections = []
    for idx, m in enumerate(matches):
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(note_text)
        sections.append((_pri(m.group(1)), note_text[m.start():end].strip()))
    sections.sort(key=lambda x: x[0])
    parts, remaining = [], max_chars
    for _, content in sections:
        if remaining <= 0:
            break
        parts.append(content[:remaining])
        remaining -= len(parts[-1])
    return '\n\n'.join(parts)


def _parse_reset_seconds(reset_str):
    if not reset_str:
        return 0.0
    try:
        m = _re.match(r'(?:(\d+\.?\d*)m)?(\d+\.?\d*)s?$', reset_str.strip())
        if m:
            return (float(m.group(1) or 0) * 60) + float(m.group(2) or 0)
    except Exception:
        pass
    return 0.0


class _GlobalRateLimiter:
    def __init__(self, max_per_minute):
        self._interval = 60.0 / max(max_per_minute, 1)
        self._last = 0.0
        self._lock = threading.Lock()

    def acquire(self):
        with self._lock:
            elapsed = time.time() - self._last
            wait = self._interval - elapsed
            if wait > 0:
                time.sleep(wait)
            self._last = time.time()


class _KeyPool:
    def __init__(self, api_keys, per_key_rpm=30):
        self._keys = api_keys
        self._n = len(api_keys)
        self._available_at = [0.0] * self._n
        self._interval = 60.0 / max(per_key_rpm, 1)
        self._lock = threading.Lock()

    def acquire(self, max_wait=900.0):
        deadline = time.time() + max_wait
        last_log = 0.0
        while True:
            now = time.time()
            if now > deadline:
                raise TimeoutError('All API keys throttled beyond max_wait')
            with self._lock:
                best = min(range(self._n), key=lambda i: self._available_at[i])
                wait = self._available_at[best] - now
                if wait <= 0:
                    self._available_at[best] = now + self._interval
                    return best, self._keys[best]
                sleep_s = min(wait, 5.0)
            if now - last_log > 60.0:
                print(f'    [KeyPool] All {self._n} keys throttled; soonest in {wait:.0f}s ...', flush=True)
                last_log = now
            time.sleep(sleep_s + 0.05)

    def mark_429(self, key_idx, cooldown_secs):
        with self._lock:
            self._available_at[key_idx] = max(
                self._available_at[key_idx], time.time() + cooldown_secs)


class GroqNoteClient:
    TOKEN_SAFETY_BUFFER = 400

    def __init__(self, api_keys, checkpoint_path=None,
                 global_rpm=60, workers=4, per_key_rpm=30):
        self.api_keys     = api_keys
        self._ckpt_path   = Path(checkpoint_path) if checkpoint_path else None
        self._results     = {}
        self._res_lock    = threading.Lock()
        self._done        = 0
        self._done_lock   = threading.Lock()
        self._ckpt_ctr    = 0
        self._ckpt_lock   = threading.Lock()
        self._ip_lim      = _GlobalRateLimiter(global_rpm)
        self._key_pool    = _KeyPool(api_keys, per_key_rpm)
        self._workers     = workers
        self._global_rpm  = global_rpm
        self._token_state = [{'remaining': 6_000, 'reset_at': 0.0}
                             for _ in range(len(api_keys))]

    def _groq_call(self, prompt, max_tokens=300, max_attempts=40):
        payload = {'model': GROQ_MODEL,
                   'messages': [{'role': 'user', 'content': prompt}],
                   'temperature': 0.2, 'max_tokens': max_tokens}
        for _ in range(max_attempts):
            key_idx, api_key = self._key_pool.acquire()
            state = self._token_state[key_idx]
            est_cost = len(prompt) // 4 + max_tokens + 200
            now = time.time()
            if state['remaining'] < max(est_cost, self.TOKEN_SAFETY_BUFFER) and now < state['reset_at']:
                sleep_s = state['reset_at'] - now + 0.5
                if sleep_s > 0:
                    time.sleep(sleep_s)
            self._ip_lim.acquire()
            try:
                resp = requests.post(
                    GROQ_API_URL,
                    headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
                    json=payload, timeout=60)
            except requests.RequestException as e:
                print(f'    [Key {key_idx}] Network error: {e}', flush=True)
                time.sleep(5)
                continue
            rem_s = resp.headers.get('x-ratelimit-remaining-tokens', '')
            rst_s = resp.headers.get('x-ratelimit-reset-tokens', '')
            if rem_s:
                try: state['remaining'] = int(rem_s)
                except ValueError: pass
            if rst_s:
                rs = _parse_reset_seconds(rst_s)
                if rs > 0: state['reset_at'] = time.time() + rs
            if resp.status_code == 200:
                return resp.json()['choices'][0]['message']['content'].strip()
            elif resp.status_code == 429:
                cooldown = 60.0
                try:
                    msg = resp.json().get('error', {}).get('message', '')
                    m = _re.search(r'Please try again in (\d+\.?\d*)s', msg)
                    if m: cooldown = float(m.group(1)) + 2.0
                except Exception: pass
                ra = resp.headers.get('Retry-After', '')
                if ra:
                    try: cooldown = max(cooldown, float(ra) + 1.0)
                    except ValueError: pass
                if state['reset_at'] > time.time():
                    cooldown = max(cooldown, state['reset_at'] - time.time() + 1.0)
                print(f'    [Key {key_idx}] 429 -> cooldown {cooldown:.0f}s, rotating', flush=True)
                self._key_pool.mark_429(key_idx, cooldown)
                state['remaining'] = 0
                continue
            else:
                print(f'    [Key {key_idx}] HTTP {resp.status_code}', flush=True)
                time.sleep(5)
        return ''

    def _worker(self, work_queue, total_todo, worker_idx):
        time.sleep(worker_idx * self._ip_lim._interval * 0.95)
        while True:
            try:
                hadm_id, note_text = work_queue.get_nowait()
            except _queue.Empty:
                break
            extracted = _extract_key_sections(note_text)
            summary = self._groq_call(NOTE_CLINICAL_INTENT_PROMPT.format(note_text=extracted))
            with self._res_lock:
                self._results[hadm_id] = summary
            with self._done_lock:
                self._done += 1
                done = self._done
            with self._ckpt_lock:
                self._ckpt_ctr += 1
                should_ckpt = self._ckpt_ctr >= CHECKPOINT_INTERVAL
                if should_ckpt: self._ckpt_ctr = 0
            if done % 100 == 0:
                n_valid = sum(1 for v in self._results.values() if v and len(v.strip()) >= 10)
                print(f'  Progress: {done}/{total_todo} | valid: {n_valid}', flush=True)
            if should_ckpt:
                self._save_checkpoint()
            work_queue.task_done()

    def _save_checkpoint(self):
        if not self._ckpt_path:
            return
        with self._res_lock:
            snap = dict(self._results)
        with open(self._ckpt_path, 'w', encoding='utf-8') as f:
            json.dump({str(k): v for k, v in snap.items()}, f, ensure_ascii=False, indent=2)

    def run(self, notes):
        cached = {}
        if self._ckpt_path and self._ckpt_path.exists():
            with open(self._ckpt_path, 'r', encoding='utf-8') as f:
                raw = json.load(f)
            cached = {int(k): v for k, v in raw.items() if v and len(str(v).strip()) >= 10}
            print(f'  Checkpoint: {len(cached)} valid, {len(raw)-len(cached)} empty -> retrying')

        todo = [(hid, txt) for hid, txt in notes.items() if hid not in cached]
        print(f'  To do: {len(todo):,}  |  Cached: {len(cached):,}  |'
              f'  Keys: {len(self.api_keys)}  |  Workers: {self._workers}')
        print(f'  Global cap: {self._global_rpm} RPM  |  '
              f'Est: ~{len(todo)/max(self._global_rpm,1)/60:.0f} min', flush=True)

        with self._res_lock:
            self._results.update(cached)
        if not todo:
            print('  Nothing to do.')
            return dict(cached)

        q = _queue.Queue()
        for item in todo:
            q.put(item)
        n_w = min(self._workers, len(todo))
        with ThreadPoolExecutor(max_workers=n_w) as pool:
            futures = [pool.submit(self._worker, q, len(todo), i) for i in range(n_w)]
            for fut in as_completed(futures):
                try: fut.result()
                except Exception as e: print(f'  [WARN] {e}', flush=True)

        self._save_checkpoint()
        with self._res_lock:
            result = dict(self._results)
        n_valid = sum(1 for v in result.values() if v and len(str(v).strip()) >= 10)
        print(f'  Done: {n_valid}/{len(result)} valid ({100*n_valid/max(len(result),1):.1f}%)')
        return result

print('Client code loaded.')

In [ ]:
# ── Load data and API keys ──

with open('data/processed/notes_text_mimic3.pkl', 'rb') as f:
    notes_data = pickle.load(f)
hadm_ids = notes_data['hadm_ids']
texts    = notes_data['notes']   # {int hadm_id: str text}
print(f'Notes loaded: {len(texts):,}')

with open('data/processed/user_config.yaml', 'r') as f:
    ucfg = yaml.safe_load(f)
api_keys = ucfg.get('groq_api_keys', [])
print(f'API keys loaded: {len(api_keys)}')
assert api_keys, 'No groq_api_keys found in user_config.yaml!'

notes_to_summarize = {
    int(hid): texts[hid]
    for hid in hadm_ids
    if hid in texts and texts[hid] and len(texts[hid].strip()) > 10
}
print(f'Notes to summarize: {len(notes_to_summarize):,} / {len(hadm_ids):,}')

In [ ]:
# ── Run Groq summarization ──
# Checkpoint saves every 50 notes to /kaggle/working/note_summaries_groq_intent.json
# Download that file from the Output tab when done (or if session times out).
#
# Tune global_rpm down if you see 429s immediately (try 30).
# workers=4 means 4 concurrent threads sharing all 20 keys via key rotation.

CHECKPOINT = '/kaggle/working/note_summaries_groq_intent.json'

client = GroqNoteClient(
    api_keys,
    checkpoint_path=CHECKPOINT,
    global_rpm=60,   # lower to 30 if 429s keep firing
    workers=4,
    per_key_rpm=30,
)

summaries = client.run(notes_to_summarize)
print(f'\nCheckpoint saved to: {CHECKPOINT}')
print('Download it from the Output tab on the right.')

In [ ]:
# ── Verify results ──

n_valid = sum(1 for v in summaries.values() if v and len(str(v).strip()) >= 10)
n_total = len(summaries)
print(f'Valid summaries : {n_valid:,} / {n_total:,} ({100*n_valid/max(n_total,1):.1f}%)')
print(f'Missing         : {len(notes_to_summarize) - n_valid:,}')

# Show a sample
sample = [(k, v) for k, v in summaries.items() if v and len(v.strip()) >= 10]
if sample:
    hid, summary = sample[0]
    print(f'\n--- Sample hadm_id={hid} ---')
    print(summary[:500])